In [ ]:
import numpy as np

In [ ]:
from sklearn.datasets import make_moons

In [ ]:
def initialize_parameters(layer_dims):
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l - 1]) * np.sqrt(2 / layer_dims[l - 1])
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [ ]:
def initialize_momentum(layer_dims, parameters):
    v = {}
    L = len(parameters) // 2
    for l in range(1, L + 1):
        v['dW' + str(l)] = np.zeros_like(parameters['W' + str(l)])
        v['db' + str(l)] = np.zeros_like(parameters['b' + str(l)])
    return v

In [ ]:
def momentum(parameters, grads, v, beta, learning_rate, layer_dims):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        v['dW' + str(l)] = v['dW' + str(l)] * beta + (1 - beta)*grads['dW' + str(l)]
        v['db' + str(l)] = v['db' + str(l)] * beta + (1 - beta)*grads['db' + str(l)]

        parameters['W' + str(l)] = parameters['W' + str(l)] - learning_rate * v['dW' + str(l)]
        parameters['b' + str(l)] = parameters['b' + str(l)] - learning_rate * v['db' + str(l)]
    return parameters, v

In [ ]:
def initialize_RMSProp(layer_dims, parameters):
    s = {}
    L = len(layer_dims) // 2
    for l in range(1, L + 1):
        s['dW' + str(l)] = np.zeros_like(parameters['W' + str(l)])
        s['db' + str(l)] = np.zeros_like(parameters['b' + str(l)])
    return s

In [ ]:
def RMSProp(layer_dims, parameters, beta, learning_rate, grads, s, epsilon = 10e-8):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        s['dW' + str(l)] = s['dW' + str(l)] * beta + (1 - beta) * np.square(grads['dW' + str(l)])
        s['db' + str(l)] = s['db' + str(l)] * beta + (1 - beta) * np.square(grads['db' + str(l)])

        parameters['W' + str(l)] = parameters['W' + str(l)] - learning_rate*(grads['dW' + str(l)])/((np.sqrt(s['dW' + str(l)])) + epsilon)
        parameters['b' + str(l)] = parameters['b' + str(l)] - learning_rate*(grads['db' + str(l)])/((np.sqrt(s['db' + str(l)])) + epsilon)
    return parameters, s

In [ ]:
def initialize_Adam(layer_dims, parameters):
    v = {}
    s = {}
    L = len(parameters) // 2
    for l in range(1, L + 1):
        v['dW' + str(l)] = np.zeros_like(parameters['W' + str(l)])
        v['db' + str(l)] = np.zeros_like(parameters['b' + str(l)])
        s['dW' + str(l)] = np.zeros_like(parameters['W' + str(l)])
        s['db' + str(l)] = np.zeros_like(parameters['b' + str(l)])
    return v, s


In [ ]:
def Adam(layer_dims, beta1, beta2, parameters, grads, v, s, t, leearning_rate, epsilon = 10e-8):
    L = len(parameters) // 2
    v_corrected = {}
    s_corrected = {}
    for l in range(1, L + 1):
        v['dW' + str(l)] = beta1*v['dW' + str(l)] + (1 - beta1)*grads['dW' + str(l)]
        v['db' + str(l)] = beta1*v['db' + str(l)] + (1 - beta1)*grads['db' + str(l)]
        s['dW' + str(l)] = beta2*s['dW' + str(l)] + (1 - beta2)*np.square(grads['dW' + str(l)])
        s['db' + str(l)] = beta2*s['db' + str(l)] + (1 - beta2)*np.square(grads['db' + str(l)])

        v_corrected['dW' + str(l)] = v['dW' + str(l)] / (1 - beta1**t)
        v_corrected['db' + str(l)] = v['db' + str(l)] / (1 - beta1**t)
        s_corrected['dW' + str(l)] = s['dW' + str(l)] / (1 - beta2**t)
        s_corrected['db' + str(l)] = s['db' + str(l)] / (1 - beta2**t)
    
        parameters['W' + str(l)] -= leearning_rate*(v_corrected['dW' + str(l)] / (np.sqrt(s_corrected['dW' + str(l)]) + epsilon))
        parameters['b' + str(l)] -= leearning_rate*(v_corrected['db' + str(l)] / (np.sqrt(s_corrected['db' + str(l)]) + epsilon))

    return parameters, v, s

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

In [ ]:
def sigmoid(Z):
    return 1 / (1+np.exp(-Z))

In [ ]:
def forward_pass(X, parameters):
    cache = {}
    cache['A0'] = X
    L = len(parameters) // 2
    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], cache['A' + str(l - 1)]) + parameters['b' + str(l)]
        cache['Z' + str(l)] = Z
        cache['A' + str(l)] = relu(Z)
    ZL = np.dot(parameters['W' + str(L)], cache['A' + str(L - 1)]) + parameters['b' + str(L)]
    cache['A' + str(L)] = sigmoid(ZL)
    cache['Z' + str(L)] = ZL
    return cache['A' + str(L)], cache 

In [ ]:
def backprop(X, Y, parameters, cache):
    m = Y.shape[1]
    L = len(parameters) // 2
    grads = {}
    dZ = cache['A' + str(L)] - Y
    grads['dW' + str(L)] = (1.0 / m) * np.dot(dZ, cache['A' + str(L-1)].T)
    grads['db' + str(L)] = (1.0 / m) * np.sum(dZ, axis = 1, keepdims = True)
    for l in reversed(range(1, L)):
        dA = np.dot(parameters['W' + str(l + 1)].T, dZ)
        dZ = np.array(dA, copy = True)
        dZ[cache['Z' + str(l)] <= 0] = 0  
        grads['dW' + str(l)] = (1.0 / m) * np.dot(dZ, cache['A' + str(l-1)].T)
        grads['db' + str(l)] = (1.0 / m) * np.sum(dZ, axis = 1, keepdims = True)
    return grads

In [ ]:
def compute_cost(AL, Y):
    m = Y.shape[1]
    AL = np.clip(AL, 1e-15, 1.0 - 1e-15)
    cross_entropy_cost = -(1.0/m) * np.sum(Y * np.log(AL) + (1.0 - Y)*np.log(1.0 - Y))
    return np.squeeze(cross_entropy_cost)

In [ ]:
from sklearn.model_selection import train_test_split
X_raw, Y_raw = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train_raw, X_test_raw, Y_train_raw, Y_test_raw = train_test_split(X_raw, Y_raw, test_size = 0.2, random_state = 42)
X_train = X_train_raw.T
X_test = X_test_raw.T
Y_train = Y_train_raw.reshape(1, -1)
Y_test = Y_test_raw.reshape(1, -1)
print(f'Train Set Shapes - X: {X_train.shape}, Y: {Y_train.shape}')
print(f'Test Set Shapes - X: {X_test.shape}, Y: {Y_test.shape}')

In [ ]:
params = initialize_parameters(layer_dims = [X_train.shape[0], 20, 10, 1])

In [ ]:
forward_pass(X_train, params)

In [ ]:
def model(X, Y, layer_dims, optimizer, learning_rate = 0.01, iter = 3000, beta1 = 0.9, beta2 = 0.999, epsilon = 10e-8):
    L = len(layer_dims)
    costs = []
    t = 0

    parameters = initialize_parameters(layer_dims)

    if optimizer == 'momentum':
        v = initialize_momentum(parameters)
    elif optimizer == 'RMSProp':
        s = initialize_RMSProp(parameters)
    elif optimizer == 'Adam':
        v, s = initialize_Adam(parameters)
    
    for i in range(iter):
        AL, cache = forward_pass(X_train, parameters)
        cost = compute_cost(AL, Y)
        grads = backprop(AL, Y, parameters, cache)

        if optimizer == 'momentum':
            parameters, v = momentum(parameters, grads, v,beta1, learning_rate, layer_dims)
        elif optimizer == 'RMSProp':
            parameters, s = RMSProp(layer_dims, parameters, beta1, learning_rate, grads, s, epsilon)
        elif optimizer == 'Adam':
            t += 1
            parameters, v, s = Adam(layer_dims, beta1, beta2, parameters, grads, v ,s, t, learning_rate, epsilon)
        
        if i%100 == 0:
            costs.append(cost)
            print(f'Cost value after iteration {i} is {cost}')
    
    return parameters, costs

In [ ]:
parameters, costs = model(
    X_train, Y_train, layer_dims = [X_train.shape[0], 20, 10, 5, 1], 
    optimizer = Adam,
    learning_rate=0.3, 
    iter=3000, 
)